# 03 - Causal analysis on the 600-founder file
Hypotheses
  H1  Age is positively associated with success up to middle age
  H2  Studying at an Ivy League school benefits success
  H3  More / higher degrees (master's, PhD) lead to success
  H4  The result depends on how success is defined
  H5  3+ years of relevant industry experience ~2x more likely to succeed (Azoulay et al. 2020)
  H6  Some age variety in the founding team helps, too much hurts (Kanama et al. 2025)

Put the three input files next to this script (or set DATA_DIR):
  PB_scraping_600_founder_age_estimates_with_web_search.xlsx   (your completed ages)
  PB_scraping_20261121.xlsx                                    (GP's scrape: schools, jobs, location)
  founders_linkedin_filled_final_with_kaggle_PB_ages.xlsx      (main dataset: industry, status)

pip install pandas openpyxl numpy scipy statsmodels scikit-learn matplotlib
optional: pip install bambi pymc arviz dowhy

## 0. Setup

In [ ]:
import os, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
if os.environ.get("SMOKE"):
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportion_confint
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")
DATA = Path(os.environ.get("DATA_DIR", "."))
OUT = Path("results"); OUT.mkdir(exist_ok=True)
AGES_FILE = "PB_scraping_600_founder_age_estimates_with_web_search.xlsx"
PB_FILE = "PB_scraping_20261121.xlsx"
MAIN_FILE = "founders_linkedin_filled_final_with_kaggle_PB_ages.xlsx"
if not (DATA / MAIN_FILE).exists():   # your copy may have a different name, e.g. "... (1).xlsx" or without _PB_ages
    cands = sorted(p.name for p in DATA.glob("founders_linkedin_filled_final_with_kaggle*.xlsx") if not p.name.startswith("~$"))
    assert cands, "Could not find the main dataset: put a file named founders_linkedin_filled_final_with_kaggle*.xlsx in this folder"
    MAIN_FILE = cands[-1]
    print("using main dataset file:", MAIN_FILE)

SEED = 42
N_BOOT = int(os.environ.get("N_BOOT", 300))   # lower it (e.g. 100) for a quick first run
AGE_MIN, AGE_MAX = 18, 60                      # ages outside this are treated as implausible
KNOT = 40                                      # "middle age" turning point used for H1 piecewise slopes
EXP_YEARS = 3                                  # H5 threshold (Azoulay et al.)
rng = np.random.default_rng(SEED)

## 1. Load and merge

In [ ]:
ages = pd.read_excel(DATA / AGES_FILE)
pb = pd.read_excel(DATA / PB_FILE)
main_raw = pd.read_excel(DATA / MAIN_FILE, sheet_name=0)

# the ages file was built from the scrape in the same row order: check it
clean_url = lambda s: s.fillna("").str.lower().str.rstrip("/")
assert len(ages) == len(pb), "ages file and scrape have different row counts"
assert (clean_url(ages.LIN_url).values == clean_url(pb.LIN_url).values).all(), \
    "row order differs between the ages file and the scrape (do not sort either file)"

# main dataset, by column position (header cells in this sheet are messy)
main = main_raw.iloc[:, [0, 1, 2, 3, 4, 5, 9, 10, 11, 12, 19]].copy()
main.columns = ["fullName", "company_name", "industry", "founding_year", "batch", "LIN_url",
                "J_verified_year", "K_bach_grad", "L_pub_age", "M_year_pub", "status"]
main["main_row"] = np.arange(len(main)) + 2      # Excel row number

df = ages.reset_index(drop=True).join(pb.add_prefix("pb_").reset_index(drop=True))
df = df.merge(main[["main_row", "industry", "batch", "status"]],
              left_on="dataset_row", right_on="main_row", how="left")

df["age"] = df["Estimated Age at Founding"].astype(float)
df["fy"] = df["Founding Year"].astype(float)
method = df["Age Method"].fillna("")
df["low_conf"] = method.str.contains("low confidence", case=False)
df["age_manual"] = method.eq("Google Search")
df["age_implausible"] = ~df.age.between(AGE_MIN, AGE_MAX)
df["age_c"] = (df.age - 30) / 10        # centred, in decades
df["age_c2"] = df.age_c ** 2

if os.environ.get("SMOKE"):               # testing hook only: scrambles the outcome
    df["status"] = df["status"].sample(frac=1, random_state=0).values

print(f"rows: {len(df)} | with age: {df.age.notna().sum()} | implausible age: {df.age_implausible.sum()} "
      f"| low-confidence age: {df.low_conf.sum()} | manual (Google) age: {df.age_manual.sum()}")

## 2. Outcomes (H4 uses all of them; the primary one is A)

In [ ]:
s = df.status.str.lower().str.strip()
is_exit = s.isin(["acquired", "public"])
def outcome(pos, neg):
    y = pd.Series(np.nan, index=df.index)
    y[s.isin(pos)] = 1
    y[s.isin(neg)] = 0
    return y

OUTCOMES = {
    "A. Exit vs inactive (active dropped)": outcome(["acquired", "public"], ["inactive"]),
    "B. Exit vs everything else (active = 0)": outcome(["acquired", "public"], ["inactive", "active"]),
    "C. IPO only": outcome(["public"], ["inactive", "active", "acquired"]),
    "D. Acquired only (IPO dropped)": outcome(["acquired"], ["inactive", "active"]),
    "E. Survival (not inactive)": outcome(["acquired", "public", "active"], ["inactive"]),
}
PRIMARY = "A. Exit vs inactive (active dropped)"
df["success"] = OUTCOMES[PRIMARY]
print("\nstatus counts:\n", s.value_counts(dropna=False).to_string())
print("\nrows available per outcome definition:")
for k, v in OUTCOMES.items():
    print(f"  {k:42s} n={v.notna().sum():4d}  success rate={v.mean():.3f}")

## 3. Controls: sector, founding period, region

In [ ]:
SECTORS = [  # (sector, regex on the main-sheet industry label)
    ("fintech_finance", r"fintech|financ|bank|insur|payment|crypto|blockchain|invest|lending"),
    ("health_bio", r"health|bio|medic|pharma|wellness|clinical|genom|care"),
    ("education", r"educat|learning|school|student"),
    ("commerce_marketplace", r"commerce|marketplace|retail|shopping|fashion|food|travel|hospitality|real estate|consumer"),
    ("media_social_games", r"media|social|messag|advertis|music|game|video|entertain|content|curated web|design"),
    ("software_infra", r"software|saas|developer|analytics|artif|machine|ai\b|cloud|security|hardware|mobile|enterprise|data|b2b"),
]
def sector_of(x):
    x = str(x).lower()
    for name, rx in SECTORS:
        if re.search(rx, x):
            return name
    return "other"
df["sector"] = df.industry.map(sector_of)
df["period"] = pd.cut(df.fy, [0, 2010, 2015, 2019, 2100],
                      labels=["<=2010", "2011-2015", "2016-2019", "2020+"]).astype(object).fillna("unknown")

US_RX = re.compile(r"united states|usa|bay area|new york|california|massachusetts|texas|boston|seattle|austin|chicago|los angeles|washington", re.I)
def region(loc):
    if pd.isna(loc): return "unknown"
    return "US" if US_RX.search(str(loc)) else "non-US"
df["region"] = df.pb_location.map(region)
CONTROLS = "C(sector) + C(period) + C(region)"
print("\nsector:", df.sector.value_counts().to_dict())
print("period:", df.period.value_counts().to_dict())
print("region:", df.region.value_counts().to_dict())

## 4. Education features (H2, H3) from the scrape

In [ ]:
NONDEG = re.compile(r"y combinator|techstars|singularity|udacity|coursera|edx|general assembly|hack reactor|bootcamp|"
                    r"nanodegree|executive education|accelerator|incubator|udemy|codecademy|fellows|thiel", re.I)
IVY_YES = re.compile(r"harvard|yale|princeton|columbia (university|college|business|engineering|law)|^columbia$|"
                     r"university of pennsylvania|upenn|wharton|^penn$|brown university|cornell|dartmouth college|^dartmouth$", re.I)
IVY_NO = re.compile(r"british columbia|penn state|pennsylvania state|massachusetts dartmouth|columbia (college chicago|basin)", re.I)
DOC = re.compile(r"phd|ph\.d|doctor|dphil|\bmd\b|\bjd\b|juris|medicine", re.I)
MAST = re.compile(r"master|\bmba\b|\bms\b|m\.s\b|\bmsc\b|\bma\b|m\.a\b|meng|\bmfa\b|mphil|\bmph\b|\bllm\b|mtech|masc|mfin", re.I)
BACH = re.compile(r"bachelor|baccalaur|bacharel|licenciad|undergrad|\bbs\b|b\.s|\bba\b|b\.a\b|\bbsc\b|\bbe\b|b\.e\b|btech|\bbeng\b|\bbba\b|\bbcom\b|s\.b\b|\bab\b|b\.sc", re.I)

def edu_entries(r):
    out = []
    for pre in ["", "Previous"]:
        name = r.get(f"pb_linkedin{pre}SchoolName")
        if pd.isna(name) or NONDEG.search(str(name)):
            continue
        deg, fld = r.get(f"pb_linkedin{pre}SchoolDegree"), r.get(f"pb_linkedin{pre}SchoolFieldOfStudy")
        out.append((str(name), "" if pd.isna(deg) else str(deg), "" if pd.isna(fld) else str(fld)))
    return out

def level(name, deg, fld):
    t = f"{deg} {fld}"
    if DOC.search(t): return 3
    if MAST.search(t) or re.search(r"graduate school of business|business school", name, re.I): return 2
    if BACH.search(t): return 1
    return 0   # school listed, degree unknown

def edu_features(r):
    es = edu_entries(r)
    lv = [level(*e) for e in es]
    ivy = any(IVY_YES.search(e[0]) and not IVY_NO.search(e[0]) for e in es)
    return pd.Series({"ivy": float(ivy) if es else np.nan,
                      "deg_level": max(lv) if lv and max(lv) > 0 else np.nan,
                      "n_degrees": sum(l > 0 for l in lv) if lv else np.nan})
edu = df.apply(edu_features, axis=1)
df = pd.concat([df, edu], axis=1)
df["has_grad"] = np.where(df.deg_level.isna(), np.nan, (df.deg_level >= 2).astype(float))
df["deg_label"] = df.deg_level.map({1: "bachelor", 2: "master", 3: "doctoral"})
print("\nIvy:", df.ivy.value_counts(dropna=False).to_dict())
print("highest degree:", df.deg_label.value_counts(dropna=False).to_dict())
print("NOTE: the scrape keeps only 2 schools per person, so higher degrees are undercounted.")

## 5. Experience features (H5) from the scrape's current + previous job

In [ ]:
def parse_range(x):
    if pd.isna(x): return (None, None)
    ys = [int(y) for y in re.findall(r"(?:19|20)\d\d", str(x))]
    if not ys: return (None, None)
    start = ys[0]
    end = None if re.search("present", str(x), re.I) else ys[-1]
    return (start, end)

norm = lambda t: re.sub(r"[^a-z0-9]", "", str(t).lower())
REL = {  # does a past job look like it is in the startup's sector?
    "software_infra": r"software|engineer|developer|\bdata\b|machine learning|\bml\b|cloud|saas|platform|infrastructure|devops|\bcto\b|technolog|\bit\b",
    "fintech_finance": r"financ|bank|invest|capital|payment|credit|insur|trading|fintech|goldman|morgan|stripe|paypal|quant",
    "health_bio": r"health|medic|clinical|pharma|\bbio|hospital|patient|life science|physician|biotech",
    "education": r"educat|school|teach|universit|learning|tutor|student|academic|course",
    "commerce_marketplace": r"commerce|retail|marketplace|merchandis|consumer|shopping|amazon|logistic|supply chain|travel|hospitality|real estate",
    "media_social_games": r"advertis|marketing|media|brand|content|music|game|gaming|entertain|social|design|video",
}
def exp_features(r):
    fy = r.fy
    if pd.isna(fy): return pd.Series({"exp_known": 0, "exp_any_years": np.nan, "exp_rel_years": np.nan})
    jobs = []
    for pre, cname in [("Previous", r.get("pb_previousCompanyName")), ("", r.get("pb_companyName"))]:
        start, end = parse_range(r.get(f"pb_linkedin{pre}JobDateRange"))
        if start is None or start >= fy:
            continue
        if pre == "" and pd.notna(cname) and (norm(cname) in norm(r.company_in_dataset) or norm(r.company_in_dataset) in norm(cname)):
            continue   # current job is the startup itself
        text = " ".join(str(x) for x in [r.get(f"pb_linkedin{pre}JobTitle"), cname, r.get(f"pb_linkedin{pre}JobDescription")] if pd.notna(x))
        dur = max(0, min(end if end else fy, fy) - start)
        rel = bool(re.search(REL.get(r.sector, r"$^"), text, re.I)) if r.sector in REL else np.nan
        jobs.append((start, dur, rel))
    if not jobs:
        return pd.Series({"exp_known": 0, "exp_any_years": np.nan, "exp_rel_years": np.nan})
    any_years = fy - min(j[0] for j in jobs)
    rel_years = np.nan if r.sector not in REL else sum(j[1] for j in jobs if j[2])
    return pd.Series({"exp_known": 1, "exp_any_years": any_years, "exp_rel_years": rel_years})
ex = df.apply(exp_features, axis=1)
df = pd.concat([df, ex], axis=1)
df["exp3"] = np.where(df.exp_rel_years.isna(), np.nan, (df.exp_rel_years >= EXP_YEARS).astype(float))
print(f"\nfounders with a usable prior job before founding: {int(df.exp_known.sum())}; "
      f"with the relevant-experience flag: {df.exp3.notna().sum()}; exp3=1: {int((df.exp3 == 1).sum())}")
# export a random sample so you can hand-check the "relevant" label before trusting H5
chk_cols = ["fullName", "company_in_dataset", "industry", "sector", "fy", "pb_previousCompanyName",
            "pb_linkedinPreviousJobTitle", "pb_linkedinPreviousJobDateRange", "exp_rel_years", "exp3"]
df[df.exp_known == 1][chk_cols].sample(min(50, int(df.exp_known.sum())), random_state=SEED).to_csv(OUT / "h5_label_check_sample.csv", index=False)

## 6. Helpers

In [ ]:
def fit(formula, data):
    return smf.glm(formula, data=data, family=sm.families.Binomial()).fit(cov_type="HC1")

def or_table(m, keep=None):
    t = pd.DataFrame({"OR": np.exp(m.params), "lo": np.exp(m.conf_int()[0]), "hi": np.exp(m.conf_int()[1]), "p": m.pvalues})
    t = t[~t.index.str.startswith("Intercept")]
    if keep: t = t[t.index.str.contains("|".join(keep))]
    return t.round(3)

def boot(data, stat, B=N_BOOT):
    vals = []
    for _ in range(B):
        try:
            v = stat(data.sample(len(data), replace=True, random_state=int(rng.integers(1e9))))
            vals.append(v)
        except Exception:
            pass
    return np.array(vals, dtype=float)

def gcomp(data, formula, treat):
    """Standardised risk under treat=1 and treat=0 -> RD and RR."""
    m = fit(formula, data)
    p1 = m.predict(data.assign(**{treat: 1})).mean()
    p0 = m.predict(data.assign(**{treat: 0})).mean()
    return p1, p0, p1 - p0, p1 / p0

def gcomp_report(data, formula, treat, label):
    p1, p0, rd, rr = gcomp(data, formula, treat)
    bs = boot(data, lambda d: gcomp(d, formula, treat)[2:])
    lo_rd, hi_rd = np.nanpercentile(bs[:, 0], [2.5, 97.5]); lo_rr, hi_rr = np.nanpercentile(bs[:, 1], [2.5, 97.5])
    print(f"{label}: risk treated={p1:.3f}, untreated={p0:.3f} | RD={rd:+.3f} [{lo_rd:+.3f},{hi_rd:+.3f}] | RR={rr:.2f} [{lo_rr:.2f},{hi_rr:.2f}] | E-value={evalue(rr):.2f} (CI limit {evalue(lo_rr if rr >= 1 else hi_rr):.2f})")
    return dict(label=label, p_treated=p1, p_untreated=p0, RD=rd, RD_lo=lo_rd, RD_hi=hi_rd, RR=rr, RR_lo=lo_rr, RR_hi=hi_rr, n=len(data))

def evalue(rr):
    """VanderWeele-Ding E-value: how strong an unmeasured confounder must be (on the RR scale) to explain the result away."""
    if not np.isfinite(rr) or rr <= 0: return np.nan
    rr = 1 / rr if rr < 1 else rr
    return rr + np.sqrt(rr * (rr - 1))

def design(data, num, cat):
    X = pd.concat([data[num].astype(float), pd.get_dummies(data[cat], drop_first=True).astype(float)], axis=1)
    return X.fillna(X.mean())

def ps_methods(data, treat, y, num, cat, caliper_sd=0.2):
    """IPW (Hajek), AIPW (doubly robust) ATE and 1:1 nearest-neighbour matching ATT on the propensity score."""
    d = data.dropna(subset=[treat, y]).reset_index(drop=True)
    X = design(d, num, cat).values; t = d[treat].values; yy = d[y].values
    ps = LogisticRegression(C=1.0, max_iter=2000).fit(X, t).predict_proba(X)[:, 1].clip(0.03, 0.97)
    ipw = (t * yy / ps).sum() / (t / ps).sum() - ((1 - t) * yy / (1 - ps)).sum() / ((1 - t) / (1 - ps)).sum()
    m1 = LogisticRegression(C=0.5, max_iter=2000).fit(X[t == 1], yy[t == 1]).predict_proba(X)[:, 1]
    m0 = LogisticRegression(C=0.5, max_iter=2000).fit(X[t == 0], yy[t == 0]).predict_proba(X)[:, 1]
    aipw = np.mean(m1 - m0 + t * (yy - m1) / ps - (1 - t) * (yy - m0) / (1 - ps))
    lps = np.log(ps / (1 - ps)); cal = caliper_sd * lps.std()
    nn = NearestNeighbors(n_neighbors=1).fit(lps[t == 0].reshape(-1, 1))
    dist, idx = nn.kneighbors(lps[t == 1].reshape(-1, 1))
    ok = dist[:, 0] <= cal
    att = (yy[t == 1][ok] - yy[t == 0][idx[ok, 0]]).mean() if ok.any() else np.nan
    return ipw, aipw, att, ps, t, X, ok.sum()

def ps_report(data, treat, y, num, cat, label):
    ipw, aipw, att, ps, t, X, nm = ps_methods(data, treat, y, num, cat)
    bs = boot(data.dropna(subset=[treat, y]), lambda d: ps_methods(d, treat, y, num, cat)[:3])
    ci = np.nanpercentile(bs, [2.5, 97.5], axis=0)
    print(f"{label}\n  IPW ATE  = {ipw:+.3f} [{ci[0,0]:+.3f},{ci[1,0]:+.3f}]\n  AIPW ATE = {aipw:+.3f} [{ci[0,1]:+.3f},{ci[1,1]:+.3f}]"
          f"\n  Matched ATT = {att:+.3f} [{ci[0,2]:+.3f},{ci[1,2]:+.3f}] (matched treated: {nm} of {int(t.sum())})")
    # covariate balance before / after weighting
    w = np.where(t == 1, 1 / ps, 1 / (1 - ps))
    smd = lambda x, w_: (np.average(x[t == 1], weights=w_[t == 1]) - np.average(x[t == 0], weights=w_[t == 0])) / np.sqrt((x[t == 1].var() + x[t == 0].var()) / 2 + 1e-9)
    names = list(design(data.dropna(subset=[treat, y]), num, cat).columns)
    bal = pd.DataFrame({"SMD_before": [smd(X[:, j], np.ones(len(t))) for j in range(X.shape[1])],
                        "SMD_after_IPW": [smd(X[:, j], w) for j in range(X.shape[1])]}, index=names).round(3)
    bal.to_csv(OUT / f"balance_{treat}.csv")
    print(f"  worst |SMD| before {bal.SMD_before.abs().max():.2f} -> after weighting {bal.SMD_after_IPW.abs().max():.2f} (aim for < 0.1 to 0.2)")
    return dict(label=label, IPW=ipw, AIPW=aipw, ATT_match=att)

def base(extra_mask=None, y="success", age_ok=True):
    d = df.copy()
    if age_ok:
        d = d[~d.age_implausible]
    d = d[d[y].notna()] if y in d else d
    return d if extra_mask is None else d[extra_mask(d)]

## 7. H1 - age and success

In [ ]:
d1 = base().copy()
print(f"\n=== H1: age (n={len(d1)}, success rate {d1.success.mean():.3f}) ===")
m1 = fit(f"success ~ age_c + age_c2 + {CONTROLS}", d1)
print(or_table(m1, ["age"]).to_string())
b1, b2 = m1.params["age_c"], m1.params["age_c2"]
tp = lambda b1, b2: 30 + 10 * (-b1 / (2 * b2)) if b2 != 0 else np.nan
tp_boot = boot(d1, lambda d: tp(*fit(f"success ~ age_c + age_c2 + {CONTROLS}", d).params[["age_c", "age_c2"]]))
print(f"quadratic turning point: age {tp(b1, b2):.1f} (bootstrap 95% CI {np.nanpercentile(tp_boot, 2.5):.1f} to {np.nanpercentile(tp_boot, 97.5):.1f}); "
      f"{'inverted-U' if b2 < 0 else 'U-shape'} (b2={b2:.3f}, p={m1.pvalues['age_c2']:.3f})")

# "up to middle age": separate slopes below / above the knot
d1["age_lo"] = (np.minimum(d1.age, KNOT) - 30) / 10
d1["age_hi"] = np.maximum(d1.age - KNOT, 0) / 10
m1b = fit(f"success ~ age_lo + age_hi + {CONTROLS}", d1)
print(f"\npiecewise slopes (knot {KNOT}), odds ratio per +10 years:")
print(or_table(m1b, ["age_lo", "age_hi"]).to_string())

# flexible shape (spline) and standardised curve
m1s = fit(f"success ~ bs(age, df=4) + {CONTROLS}", d1)
grid = np.arange(20, 56, 1)
def std_curve(data, formula=f"success ~ bs(age, df=4) + {CONTROLS}"):
    m = fit(formula, data)
    return np.array([m.predict(data.assign(age=a)).mean() for a in grid])
curve = std_curve(d1)
bc = boot(d1, std_curve, B=min(N_BOOT, 200))
lo, hi = np.nanpercentile(bc, [2.5, 97.5], axis=0)
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(grid, curve, color="#1f4e79"); ax.fill_between(grid, lo, hi, alpha=0.2, color="#1f4e79")
bands = pd.cut(d1.age, [17, 24, 29, 34, 39, 44, 60]); g = d1.groupby(bands).success.agg(["mean", "count"])
ax.scatter([a.mid for a in g.index], g["mean"], color="#c0392b", zorder=3, label="raw rate by age band")
ax.set_xlabel("age at founding"); ax.set_ylabel("P(exit)"); ax.set_title("H1: adjusted success probability by age"); ax.legend()
fig.tight_layout(); fig.savefig(OUT / "h1_age_curve.png", dpi=200); plt.show()
print("raw success rate by age band:\n", g.round(3).to_string())

## 8. H2 - Ivy League

In [ ]:
d2 = base(lambda d: d.ivy.notna()).copy()
print(f"\n=== H2: Ivy League (n={len(d2)}, Ivy={int(d2.ivy.sum())}, raw success: Ivy {d2[d2.ivy==1].success.mean():.3f} vs other {d2[d2.ivy==0].success.mean():.3f}) ===")
print("Total effect (no age control):")
print(or_table(fit(f"success ~ ivy + {CONTROLS}", d2), ["ivy"]).to_string())
print("Adjusted for age too:")
print(or_table(fit(f"success ~ ivy + age_c + age_c2 + {CONTROLS}", d2), ["ivy"]).to_string())
r2 = [gcomp_report(d2, f"success ~ ivy + {CONTROLS}", "ivy", "Ivy (g-computation, no age)"),
      gcomp_report(d2, f"success ~ ivy + age_c + age_c2 + {CONTROLS}", "ivy", "Ivy (g-computation, with age)")]
ps_report(d2, "ivy", "success", ["age_c"], ["sector", "period", "region"], "Ivy: propensity-score methods")
pd.DataFrame(r2).to_csv(OUT / "h2_ivy.csv", index=False)

## 9. H3 - higher degrees

In [ ]:
d3 = base(lambda d: d.deg_level.notna()).copy()
print(f"\n=== H3: degrees (n={len(d3)}) ===  highest degree counts: {d3.deg_label.value_counts().to_dict()}")
print(d3.groupby("deg_label").success.agg(["mean", "count"]).round(3).to_string())
m3 = fit(f'success ~ C(deg_label, Treatment("bachelor")) + {CONTROLS}', d3)
print("vs bachelor's only (no age control = total effect):"); print(or_table(m3, ["deg_label"]).to_string())
m3b = fit(f'success ~ C(deg_label, Treatment("bachelor")) + age_c + age_c2 + {CONTROLS}', d3)
print("with age control:"); print(or_table(m3b, ["deg_label"]).to_string())
r3 = [gcomp_report(d3, f"success ~ has_grad + {CONTROLS}", "has_grad", "Master's or PhD vs bachelor's (no age)"),
      gcomp_report(d3, f"success ~ has_grad + age_c + age_c2 + {CONTROLS}", "has_grad", "Master's or PhD vs bachelor's (with age)")]
ps_report(d3, "has_grad", "success", ["age_c"], ["sector", "period", "region"], "Graduate degree: propensity-score methods")
pd.DataFrame(r3).to_csv(OUT / "h3_degrees.csv", index=False)
print("Reminder: degrees raise age at founding, so the no-age model is the total effect; the with-age model is the effect beyond age.")

## 10. H4 - does the answer depend on the success definition?

In [ ]:
rows = []
for name, y in OUTCOMES.items():
    d = df[~df.age_implausible].assign(y=y).dropna(subset=["y", "ivy", "has_grad"])
    if d.y.nunique() < 2 or d.y.sum() < 10:
        print(f"skip {name}: too few successes"); continue
    try:
        m = fit(f"y ~ age_c + age_c2 + ivy + has_grad + {CONTROLS}", d)
    except Exception as e:
        print(f"skip {name}: {e}"); continue
    t = or_table(m, ["age_c", "ivy", "has_grad"]); t.insert(0, "outcome", name); t.insert(1, "n", len(d)); t.insert(2, "base_rate", round(d.y.mean(), 3))
    rows.append(t.reset_index().rename(columns={"index": "term"}))
h4 = pd.concat(rows); h4.to_csv(OUT / "h4_outcome_definitions.csv", index=False)
print("\n=== H4: same model, different success definitions (odds ratios) ===")
print(h4.to_string(index=False))
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
for ax, term in zip(axes, ["age_c", "ivy", "has_grad"]):
    t = h4[h4.term == term].reset_index(drop=True)
    ax.errorbar(t.OR, range(len(t)), xerr=[t.OR - t.lo, t.hi - t.OR], fmt="o", color="#1f4e79"); ax.axvline(1, color="grey", ls="--")
    ax.set_yticks(range(len(t))); ax.set_yticklabels(t.outcome.str.slice(0, 22)); ax.set_xscale("log")
    ax.set_title({"age_c": "age (per +10 yrs, linear term)", "ivy": "Ivy", "has_grad": "master's/PhD"}[term]); ax.invert_yaxis()
fig.tight_layout(); fig.savefig(OUT / "h4_outcome_definitions.png", dpi=200); plt.show()

## 11. H5 - relevant experience (Azoulay et al.: ~2x)

In [ ]:
d5 = base(lambda d: d.exp3.notna()).copy()
print(f"\n=== H5: relevant experience >= {EXP_YEARS} yrs (n={len(d5)}, treated={int(d5.exp3.sum())}) ===")
print("EXPLORATORY: only founders whose scrape shows a job that started before founding are included.")
raw = d5.groupby("exp3").success.agg(["mean", "sum", "count"]); print(raw.round(3).to_string())
if d5.exp3.nunique() == 2 and d5.success.sum() > 10:
    formula5 = f"success ~ exp3 + age_c + age_c2 + ivy + has_grad + {CONTROLS}"
    d5m = d5.fillna({"ivy": 0, "has_grad": 0})   # missing education treated as 'none listed' for this model
    print(or_table(fit(formula5, d5m), ["exp3"]).to_string())
    r5 = gcomp_report(d5m, formula5, "exp3", f"Relevant experience >= {EXP_YEARS} yrs")
    print("Paper's benchmark: RR about 2 (top 0.1% growth outcome, so not directly comparable to exit).")
    pd.DataFrame([r5]).to_csv(OUT / "h5_experience.csv", index=False)
    # dose-response with total years before founding (any job)
    d5["exp_any_c"] = d5.exp_any_years
    print(or_table(fit(f"success ~ exp_any_c + age_c + {CONTROLS}", d5.dropna(subset=['exp_any_c'])), ["exp_any_c"]).to_string())
else:
    print("Not enough variation or successes to estimate this. Improve the experience data first (re-scrape the full experience section).")

## 12. H6 - team age mix (Kanama et al.)

In [ ]:
# With only the 600 rows few teams have 2+ founders in the file. Set USE_FULL_DATASET = True to also pull co-founders from the main dataset.
USE_FULL_DATASET = False   # False = only the 600 founders in your file; True = use every founder in the main dataset
m_ = main.copy()
m_["fy"] = m_.J_verified_year.fillna(m_.founding_year)
pub = m_.L_pub_age.notna() & m_.M_year_pub.notna() & m_.fy.notna()
m_["age"] = np.where(pub, m_.L_pub_age + (m_.fy - m_.M_year_pub), np.where(m_.K_bach_grad.notna() & m_.fy.notna(), m_.fy - m_.K_bach_grad + 22, np.nan))
m_ = m_.merge(df[["dataset_row", "age"]].rename(columns={"age": "age600"}), left_on="main_row", right_on="dataset_row", how="left")
m_["age"] = m_.age600.fillna(m_.age)                        # your completed 600-file ages take priority
m_ = m_[m_.age.between(AGE_MIN, AGE_MAX) | m_.age.isna()]
if not USE_FULL_DATASET:
    m_ = m_[m_.dataset_row.notna()]                         # keep only the 600 founders
m_["team"] = m_.company_name.map(norm) + "_" + m_.fy.astype("Int64").astype(str)
status = m_.status.str.lower().str.strip()
m_["success"] = np.where(status.isin(["acquired", "public"]), 1, np.where(status.eq("inactive"), 0, np.nan))
teams = m_.groupby("team").agg(n_founders=("fullName", "size"), n_aged=("age", "count"), spread=("age", lambda a: a.max() - a.min()),
                               sd=("age", "std"), mean_age=("age", "mean"), success=("success", "first"), industry=("industry", "first"), fy=("fy", "first"))
t6 = teams[(teams.n_founders >= 2) & (teams.n_aged == teams.n_founders) & teams.success.notna()].copy()
t6["sector"] = t6.industry.map(sector_of)
t6["period"] = pd.cut(t6.fy, [0, 2010, 2015, 2019, 2100], labels=["<=2010", "2011-2015", "2016-2019", "2020+"]).astype(object).fillna("unknown")
t6["band"] = pd.cut(t6.spread, [-1, 2, 9, 100], labels=["0-2 yrs", "3-9 yrs", "10+ yrs"])
print(f"\n=== H6: team age mix ===\nteams with 2+ founders, all aged, outcome known: {len(t6)}; spread median {t6.spread.median():.0f}y; "
      f"10+ yr spread: {(t6.spread >= 10).sum()}; 20+ yr: {(t6.spread >= 20).sum()}")
print(t6.groupby("band").success.agg(["mean", "count"]).round(3).to_string())
print("NOTE: very few teams have a large spread, so the 'too much hurts' half cannot really be tested.")
if len(t6) > 40 and t6.success.sum() > 10:
    t6["spread_c"] = t6.spread / 10; t6["spread_c2"] = t6.spread_c ** 2
    m6 = fit("success ~ spread_c + spread_c2 + n_founders + C(sector) + C(period)", t6)
    print(or_table(m6, ["spread", "n_founders"]).to_string())
    m6b = fit("success ~ C(band) + n_founders + C(sector) + C(period)", t6)
    print(or_table(m6b, ["band", "n_founders"]).to_string())
    t6.to_csv(OUT / "h6_teams.csv")

## 13. Robustness - does the age/ivy/degree story survive stricter age data?

In [ ]:
checks = {
    "all plausible ages": lambda d: d,
    "drop low-confidence ages": lambda d: d[~d.low_conf],
    "drop manual (Google Search) ages": lambda d: d[~d.age_manual],
    "only exact / published-age rows": lambda d: d[d["Age Method"].fillna("").str.contains("Exact|published", case=False)],
    "US-located founders only": lambda d: d[d.region == "US"],
}
rb = []
for name, f in checks.items():
    d = f(base().dropna(subset=["ivy", "has_grad"]))
    if len(d) < 60 or d.success.nunique() < 2:
        continue
    try:
        m = fit(f"success ~ age_c + age_c2 + ivy + has_grad + {CONTROLS}", d)
        t = or_table(m, ["age_c", "ivy", "has_grad"]); t.insert(0, "sample", name); t.insert(1, "n", len(d))
        rb.append(t.reset_index().rename(columns={"index": "term"}))
    except Exception as e:
        print("skip", name, e)
rb = pd.concat(rb); rb.to_csv(OUT / "robustness.csv", index=False)
print("\n=== Robustness (odds ratios) ===\n", rb.to_string(index=False))

## 14. Bayesian logistic regression (optional: pip install bambi pymc arviz)

In [ ]:
try:
    import bambi as bmb
    d = base().dropna(subset=["ivy", "has_grad"]).copy()
    d["ivy"] = d.ivy.astype(int); d["has_grad"] = d.has_grad.astype(int)
    priors = {"common": bmb.Prior("Normal", mu=0, sigma=1)}          # weakly informative on the log-odds scale
    bm = bmb.Model("success ~ age_c + age_c2 + ivy + has_grad + (1|sector) + (1|period) + (1|region)", d, family="bernoulli", priors=priors)
    idata = bm.fit(draws=1500, tune=1500, chains=4, random_seed=SEED, target_accept=0.95)

    # summaries computed directly from the posterior draws, so this does not depend on the arviz version
    post = idata.posterior
    terms = ["age_c", "age_c2", "ivy", "has_grad"]
    rows = []
    for v in terms:
        x = np.asarray(post[v].values).ravel()
        lo, hi = np.percentile(x, [3, 97])               # 94% credible interval
        rows.append(dict(term=v, mean=x.mean(), sd=x.std(), cred_lo=lo, cred_hi=hi,
                         OR_median=np.exp(np.median(x)), OR_lo=np.exp(lo), OR_hi=np.exp(hi), P_effect_above_0=(x > 0).mean()))
    bayes = pd.DataFrame(rows).round(3)
    bayes.to_csv(OUT / "bayes_summary.csv", index=False)
    print(bayes.to_string(index=False))

    fig, ax = plt.subplots(figsize=(6, 3.2))
    ax.errorbar(bayes.OR_median, range(len(bayes)), xerr=[bayes.OR_median - bayes.OR_lo, bayes.OR_hi - bayes.OR_median], fmt="o", color="#1f4e79")
    ax.axvline(1, color="grey", ls="--"); ax.set_xscale("log")
    ax.set_yticks(range(len(bayes))); ax.set_yticklabels(["age (per +10 yrs)", "age squared", "Ivy", "master's/PhD"]); ax.invert_yaxis()
    ax.set_xlabel("odds ratio (posterior median and 94% credible interval)"); ax.set_title("Bayesian logistic regression")
    fig.tight_layout(); fig.savefig(OUT / "bayes_forest.png", dpi=200); plt.show()

    # sampler health check (optional; skipped quietly if the arviz version differs)
    try:
        import arviz as az
        print(az.summary(idata, var_names=terms))
        print("Look for r_hat close to 1.00 and ess_bulk in the hundreds or more.")
    except Exception as e:
        print("(skipped arviz diagnostics:", type(e).__name__, ")")
except ImportError:
    print("\n(bambi / pymc not installed: skipping the Bayesian model. pip install bambi pymc arviz)")

## 15. Optional: DAG + refutation with DoWhy (pip install dowhy)

In [ ]:
try:
    from dowhy import CausalModel
    dd = base().dropna(subset=["ivy"]).copy()
    X = pd.get_dummies(dd[["sector", "period", "region"]], drop_first=True).astype(float)
    dd = pd.concat([dd[["success", "ivy", "age_c"]], X], axis=1).dropna()
    covs = list(X.columns) + ["age_c"]
    cm = CausalModel(data=dd, treatment="ivy", outcome="success", common_causes=covs)
    est = cm.identify_effect(proceed_when_unidentifiable=True)
    ate = cm.estimate_effect(est, method_name="backdoor.propensity_score_weighting", target_units="ate")
    print("\nDoWhy IPW ATE for Ivy:", round(float(ate.value), 3))
    for ref in ["random_common_cause", "placebo_treatment_refuter", "data_subset_refuter"]:
        print(cm.refute_estimate(est, ate, method_name=ref))
except ImportError:
    print("\n(dowhy not installed: skipping)")

print("\nDone. Tables and figures are in:", OUT.resolve())